# 3. Квантильная регрессия

**Основная идея:** напрямую моделируем квантили, получая адаптивные интервалы (ширина зависит от признаков).

В этом ноутбуке:
- Pinball loss — почему её минимум даёт квантиль;
- QAR (Quantile Autoregression) через statsmodels QuantReg;
- CatBoost Quantile vs MultiQuantile, пересечение квантилей;
- CQR (Conformalized Quantile Regression) — возвращаем покрытие;
- финальное сравнение всех методов блока.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.regression.quantile_regression import QuantReg
from catboost import CatBoostRegressor
from conformalforecast import ConformalizedQuantileRegression

sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 4)

## 0. Данные

Те же AEP дневные, что и в предыдущих ноутбуках.

In [ ]:
df_hour = pd.read_csv('../data/AEP_hourly.csv', parse_dates=['Datetime'])
df_hour.columns = ['timestamp', 'target']

df_day = df_hour.resample('D', on='timestamp').sum().reset_index()
df_day = df_day.rename(columns={'timestamp': 'ds', 'target': 'y'})

split_point = int(len(df_day) * 0.8)
train = df_day.iloc[:split_point].copy()
test = df_day.iloc[split_point:].copy()

print(f"Train: {len(train)}, Test: {len(test)}")

## 1. Pinball loss

Для квантиля уровня τ:

$$L_τ(y, q) = \begin{cases} τ(y - q), & y ≥ q \\ (1-τ)(q - y), & y < q \end{cases}$$

Минимизация E[L_τ(y, q)] даёт τ-квантиль распределения y.

In [ ]:
# Демонстрация на синтетике
np.random.seed(42)
y_demo = np.random.normal(100, 15, 1000)

# Кандидаты для квантиля
q_candidates = np.linspace(50, 150, 200)

def pinball_loss(y, q, tau):
    return np.where(y >= q, tau * (y - q), (1 - tau) * (q - y))

In [ ]:
# Потери для разных τ
taus = [0.1, 0.5, 0.9]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, tau in zip(axes, taus):
    losses = [pinball_loss(y_demo, q, tau).mean() for q in q_candidates]
    q_min = q_candidates[np.argmin(losses)]

    ax.plot(q_candidates, losses)
    ax.axvline(q_min, color='red', linestyle='--', label=f'Минимум q={q_min:.1f}')
    ax.axvline(np.quantile(y_demo, tau), color='green', linestyle=':', label=f'Истинный квантиль={np.quantile(y_demo, tau):.1f}')
    ax.set_title(f'τ = {tau}')
    ax.set_xlabel('q')
    ax.set_ylabel('Pinball loss')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Минимум pinball loss совпадает с эмпирическим квантилем.")

## 2. QAR (Quantile Autoregression)

Используем statsmodels QuantReg для оценки квантилей через авторегрессию.
Модель: $q_τ(y_t) = φ_0(τ) + φ_1(τ) y_{t-1} + ... + φ_p(τ) y_{t-p}$

In [ ]:
# Признаки: лаги
def make_ar_features(series, lags=list(range(1, 8))):
    df = pd.DataFrame({'y': series.values})
    for lag in lags:
        df[f'lag_{lag}'] = df['y'].shift(lag)
    df = df.dropna()
    return df

In [ ]:
df_ar = make_ar_features(train['y'], lags=list(range(1, 15)))
X_train_ar = df_ar[[c for c in df_ar.columns if c.startswith('lag_')]]
y_train_ar = df_ar['y']

# Тестовая выборка
df_ar_test = make_ar_features(pd.concat([train['y'], test['y']]).reset_index(drop=True), lags=list(range(1, 15)))
df_ar_test = df_ar_test.iloc[len(df_ar):].reset_index(drop=True)
X_test_ar = df_ar_test[[c for c in df_ar_test.columns if c.startswith('lag_')]]
y_test_ar = df_ar_test['y']

In [ ]:
# Оценка квантилей 0.025, 0.5, 0.975
quantiles = [0.025, 0.5, 0.975]
qar_predictions = {}

for tau in quantiles:
    qr = QuantReg(y_train_ar, X_train_ar)
    qr_fit = qr.fit(q=tau)
    qar_predictions[tau] = qr_fit.predict(X_test_ar)

    print(f"\nКвантиль τ={tau}:")
    print(f"  Коэффициенты: {qr_fit.params.values[:3]}...")

In [ ]:
# Визуализация первых 60 точек
plt.figure(figsize=(14, 4))
plt.plot(range(60), y_test_ar.values[:60], 'o-', label='Факт', markersize=3)
plt.plot(range(60), qar_predictions[0.5][:60], 's-', label='Медиана (τ=0.5)', markersize=3)
plt.fill_between(range(60), qar_predictions[0.025][:60], qar_predictions[0.975][:60], alpha=0.3, label='95% QAR')
plt.legend()
plt.title('Quantile Autoregression (QAR)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Покрытие
coverage_qar = ((y_test_ar.values >= qar_predictions[0.025]) & (y_test_ar.values <= qar_predictions[0.975])).mean()
print(f"Покрытие QAR: {coverage_qar:.3f}")

### Асимметрия: φ(τ) зависит от квантиля

Лонгрид нед. 7 с. 16 содержит ошибку: τ=0.9 с наклоном 0.989 — **самый крутой**, а не самый пологий.
Большой φ₁ означает сильную зависимость от прошлого.

In [ ]:
# Визуализация коэффициентов
coef_by_tau = {tau: QuantReg(y_train_ar, X_train_ar).fit(q=tau).params['lag_1'] for tau in np.linspace(0.1, 0.9, 9)}

plt.figure(figsize=(10, 4))
plt.plot(list(coef_by_tau.keys()), list(coef_by_tau.values()), 'o-')
plt.xlabel('τ')
plt.ylabel('φ₁ (коэффициент lag_1)')
plt.title('Зависимость AR-коэффициента от квантиля')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("φ₁ растёт с τ: верхние квантили сильнее зависят от истории (инерция пиков).")

## 3. CatBoost для квантилей

CatBoost поддерживает `Quantile:alpha=...` (одна модель на квантиль) и `MultiQuantile` (одна модель на все квантили).

In [ ]:
# Признаки: лаги + календарь
def make_features_cb(df, lags=list(range(14, 28))):
    df = df.copy()
    df['dow'] = df['ds'].dt.dayofweek
    df['month'] = df['ds'].dt.month

    for lag in lags:
        df[f'lag_{lag}'] = df['y'].shift(lag)

    df = df.dropna()
    return df

In [ ]:
df_cb = make_features_cb(df_day)

X_train_cb = df_cb.iloc[:split_point - 28][['dow', 'month'] + [c for c in df_cb.columns if c.startswith('lag_')]]
y_train_cb = df_cb.iloc[:split_point - 28]['y']

X_test_cb = df_cb.iloc[split_point - 28:][X_train_cb.columns]
y_test_cb = df_cb.iloc[split_point - 28:]['y']

print(f"Train: {len(X_train_cb)}, Test: {len(X_test_cb)}")

In [ ]:
# Отдельные модели Quantile
cb_quantiles = {}

for tau in [0.025, 0.5, 0.975]:
    cb = CatBoostRegressor(
        iterations=200,
        depth=4,
        loss_function=f'Quantile:alpha={tau}',
        verbose=False,
        random_state=42
    )
    cb.fit(X_train_cb, y_train_cb)
    cb_quantiles[tau] = cb.predict(X_test_cb)

In [ ]:
# MultiQuantile: одна модель на все квантили
cb_multi = CatBoostRegressor(
    iterations=200,
    depth=4,
    loss_function='MultiQuantile:alpha=0.025,0.5,0.975',
    verbose=False,
    random_state=42
)
cb_multi.fit(X_train_cb, y_train_cb)

# Прогноз возвращает матрицу (n_samples, n_quantiles)
preds_multi = cb_multi.predict(X_test_cb)

cb_multi_quantiles = {
    0.025: preds_multi[:, 0],
    0.5: preds_multi[:, 1],
    0.975: preds_multi[:, 2]
}

In [ ]:
# Визуализация: сравнение отдельных моделей и MultiQuantile
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Отдельные модели
axes[0].plot(range(60), y_test_cb.values[:60], 'o-', label='Факт', markersize=3)
axes[0].fill_between(range(60), cb_quantiles[0.025][:60], cb_quantiles[0.975][:60], alpha=0.3, label='95% (отдельные)')
axes[0].set_title('CatBoost Quantile (три отдельные модели)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# MultiQuantile
axes[1].plot(range(60), y_test_cb.values[:60], 'o-', label='Факт', markersize=3)
axes[1].fill_between(range(60), cb_multi_quantiles[0.025][:60], cb_multi_quantiles[0.975][:60], alpha=0.3, label='95% (MultiQuantile)', color='orange')
axes[1].set_title('CatBoost MultiQuantile (одна модель)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Пересечение квантилей

Квантили должны быть упорядочены: q₀.₀₂₅ ≤ q₀.₅ ≤ q₀.₉₇₅. Отдельные модели могут нарушить это.

In [ ]:
# Проверка пересечений
violations_separate = np.sum(cb_quantiles[0.025] > cb_quantiles[0.975])
violations_multi = np.sum(cb_multi_quantiles[0.025] > cb_multi_quantiles[0.975])

print(f"Пересечений (отдельные модели): {violations_separate} из {len(y_test_cb)} ({100*violations_separate/len(y_test_cb):.1f}%)")
print(f"Пересечений (MultiQuantile): {violations_multi} из {len(y_test_cb)} ({100*violations_multi/len(y_test_cb):.1f}%)")

print("\nMultiQuantile лучше сохраняет порядок квантилей.")

## 4. CQR (Conformalized Quantile Regression)

**Проблема:** квантили, оценённые независимо, не гарантируют покрытие вне выборки (лонгрид с. 6).

**Решение:** конформная калибровка квантилей — CQR сохраняет адаптивную ширину и возвращает покрытие.

In [ ]:
# Разбиение на train / calib / test
calib_size = 100

X_train_cqr = X_train_cb.iloc[:-calib_size]
y_train_cqr = y_train_cb.iloc[:-calib_size]

X_calib_cqr = X_train_cb.iloc[-calib_size:]
y_calib_cqr = y_train_cb.iloc[-calib_size:]

In [ ]:
# Обучаем квантильные модели на train
cb_lower = CatBoostRegressor(iterations=200, depth=4, loss_function='Quantile:alpha=0.025', verbose=False, random_state=42)
cb_upper = CatBoostRegressor(iterations=200, depth=4, loss_function='Quantile:alpha=0.975', verbose=False, random_state=42)

cb_lower.fit(X_train_cqr, y_train_cqr)
cb_upper.fit(X_train_cqr, y_train_cqr)

# Прогнозы на калибровке
q_lower_calib = cb_lower.predict(X_calib_cqr)
q_upper_calib = cb_upper.predict(X_calib_cqr)

In [ ]:
# CQR через conformalforecast
cqr = ConformalizedQuantileRegression()
cqr.fit(y_calib_cqr.values, q_lower_calib, q_upper_calib)

# Прогнозы на тесте
q_lower_test = cb_lower.predict(X_test_cb)
q_upper_test = cb_upper.predict(X_test_cb)

lower_cqr, upper_cqr = cqr.predict(q_lower_test, q_upper_test, alpha=0.05)

In [ ]:
# Сравнение: квантили без калибровки vs CQR
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Без калибровки
axes[0].plot(range(60), y_test_cb.values[:60], 'o-', label='Факт', markersize=3)
axes[0].fill_between(range(60), q_lower_test[:60], q_upper_test[:60], alpha=0.3, label='Квантили без калибровки')
axes[0].set_title('CatBoost Quantile (без калибровки)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# С калибровкой (CQR)
axes[1].plot(range(60), y_test_cb.values[:60], 'o-', label='Факт', markersize=3)
axes[1].fill_between(range(60), lower_cqr[:60], upper_cqr[:60], alpha=0.3, label='CQR', color='orange')
axes[1].set_title('CQR (с конформной калибровкой)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
coverage_no_calib = ((y_test_cb.values >= q_lower_test) & (y_test_cb.values <= q_upper_test)).mean()
coverage_cqr = ((y_test_cb.values >= lower_cqr) & (y_test_cb.values <= upper_cqr)).mean()

print(f"Покрытие без калибровки: {coverage_no_calib:.3f}")
print(f"Покрытие CQR: {coverage_cqr:.3f} (номинал 0.950)")
print("\nCQR возвращает покрытие к номиналу, сохраняя адаптивную ширину.")

## 5. Финальное сравнение всех методов блока

Сравниваем coverage, width, Winkler score всех методов из трёх ноутбуков.

In [ ]:
# Хелпер для метрик
def interval_metrics(y_true, lower, upper, level=0.95):
    coverage = ((y_true >= lower) & (y_true <= upper)).mean()
    width = (upper - lower).mean()

    alpha = 1 - level
    winkler = (upper - lower) + (2/alpha) * (lower - y_true) * (y_true < lower) + (2/alpha) * (y_true - upper) * (y_true > upper)
    winkler = winkler.mean()

    return {'coverage': coverage, 'width': width, 'winkler': winkler}

In [ ]:
# Собираем результаты (используем те же прогнозы, что выше)
results = []

# QAR
results.append({
    'Метод': 'QAR (QuantReg)',
    **interval_metrics(y_test_ar.values, qar_predictions[0.025], qar_predictions[0.975])
})

# CatBoost Quantile
results.append({
    'Метод': 'CatBoost Quantile',
    **interval_metrics(y_test_cb.values, cb_quantiles[0.025], cb_quantiles[0.975])
})

# CatBoost MultiQuantile
results.append({
    'Метод': 'CatBoost MultiQuantile',
    **interval_metrics(y_test_cb.values, cb_multi_quantiles[0.025], cb_multi_quantiles[0.975])
})

# CQR
results.append({
    'Метод': 'CQR',
    **interval_metrics(y_test_cb.values, lower_cqr, upper_cqr)
})

In [ ]:
results_df = pd.DataFrame(results)
print("\nФинальное сравнение методов блока 1 (квантильная регрессия):\n")
print(results_df.to_string(index=False))

## Итоговая памятка: когда что брать

| Метод | Когда использовать | Плюсы | Минусы |
|-------|-------------------|-------|--------|
| Аналитические (ETS/ARIMA) | Модель точно подходит, ряд стационарен | Быстро, теоретически обоснованно | Занижает покрытие при неверной модели |
| NaiveVariance / Empirical | Есть кросс-валидация, модель любая | Простота, σ по горизонту | Остатки in-sample оптимистичны |
| Бутстреп остатков | i.i.d. ошибки, нужна неопределённость шума | Учитывает случайность | Не учитывает неопределённость параметров |
| Бэггинг (block bootstrap) | Автокорреляция, нужна неопределённость модели | Учитывает зависимость | Медленно |
| Split Conformal | Перестановочность, модель любая | Гарантия покрытия | Одинаковая ширина для всех точек |
| ACI / AgACI | Дрейф, гетероскедастичность | Адаптация онлайн | Сложнее настроить |
| QAR | Нужны асимметричные интервалы | Адаптивная ширина | Покрытие вне выборки не гарантировано |
| CQR | Квантильная регрессия + гарантия покрытия | Адаптивность + калибровка | Требует калибровочной выборки |

**Общий вывод блока:** для надёжных интервалов нужна калибровка на отложенных данных. Конформные методы дают гарантии при перестановочности; квантильная регрессия даёт адаптивную ширину; CQR объединяет оба подхода.